# 03 — Independently Fine-Tuned ResNet18 and RGB/Grayscale Comparison

**Research question:** How does an ImageNet-pretrained ResNet18 compare with independently trained scratch CNNs on the same four frozen train/validation/test designs? Does colour removal change scratch-CNN generalization in O and ST?

The full research profile loads ImageNet weights separately for each new ResNet18 model. Models are fine-tuned on **that regime's** training split, selected on that regime's validation set, and evaluated once on that regime's held-out test split. **No model is transferred from O to S to T to ST.** The smoke profile intentionally uses randomly initialized ResNet18 weights to avoid downloading pretrained parameters, so its scores have no scientific significance.

In [ ]:
from pathlib import Path
import sys,json,pandas as pd,matplotlib.pyplot as plt
from IPython.display import display,Image as IPyImage
ROOT=Path.cwd().resolve();sys.path.insert(0,str(ROOT/'generated'))
import monreader_methods as mr
settings=mr.load_settings(ROOT);device=mr.setup(settings['seed'],settings['smoke'])
print('ImageNet initialization:',settings['resnet_pretrained'])
print('ResNet18 trainable parameters:',sum(p.numel() for p in mr.get_resnet(pretrained=False).parameters() if p.requires_grad))

## Fine-tune four separate ResNet18 models

We use BCE-with-logits loss, ImageNet-normalized RGB input, a new one-logit classification head and **validation Macro F1** for best-checkpoint selection. The held-out test data are only used *after* checkpoint selection. Training and prediction files are generated anew by the following code.

In [ ]:
resnet_results=[]
for regime in mr.REGIMES:
    df=pd.read_csv(Path(settings['out'])/'splits'/f'{regime}_split.csv')
    result=mr.train_resnet_regime(df,regime,settings,device)
    resnet_results.append(result)
    print(regime,'complete')
rr=pd.DataFrame(resnet_results);rr.to_csv(Path(settings['out'])/'resnet18_heldout_metrics.csv',index=False)
display(rr[['regime','val_validation_macro_f1','test_macro_f1','test_fp','test_fn']])

## Compare the two model families using the **same held-out metric**

Read the scratch results generated in Notebook 02 and the ResNet18 results calculated above. The bar chart uses held-out **Macro F1**, not W&B tuning F1, and is saved as an image suitable for the README. It makes no causal assertion about why transfer learning improves or worsens particular regimes.

In [ ]:
scratch=pd.read_csv(Path(settings['out'])/'scratch_heldout_metrics.csv')
results=pd.concat([scratch,rr],ignore_index=True)
display(results[['regime','model_family','test_macro_f1','test_accuracy']])
figure=mr.plot_model_comparison(results,Path(settings['out'])/'figures'/'model_comparison.png')
display(IPyImage(filename=str(figure)))

## Controlled grayscale experiment (Scratch CNN: O and ST)

The grayscale comparison is a **new training experiment** on corresponding train/validation/test partitions, not simply an RGB model evaluated on recoloured test images. Every grayscale model uses the selected RGB architecture and training settings for its regime, but starts with new random weights. Conversion is consistent across all three roles. This is a comparison of input representation, not a guarantee that performance will increase or decrease.

In [ ]:
gray_results=[]
for regime in ('O','ST'):
    df=pd.read_csv(Path(settings['out'])/'splits'/f'{regime}_split.csv')
    gray_results.append(mr.train_grayscale(df,regime,settings,device))
gr=pd.DataFrame(gray_results);gr.to_csv(Path(settings['out'])/'grayscale_heldout_metrics.csv',index=False)
display(gr[['regime','macro_f1','accuracy']])
manifest=pd.read_csv(Path(settings['out'])/'canonical_manifest.csv')
figure=mr.plot_grayscale_examples(manifest,Path(settings['out'])/'figures'/'rgb_vs_grayscale_examples.png')
display(IPyImage(filename=str(figure)))

## Interpretation and handoff

Both model families have independently initialized/fine-tuned models **per regime**. Full probabilities and ground truths have been saved. Notebook 04 will use these per-frame outputs to estimate source-cluster uncertainty and error patterns, rather than relying on manually entered summary tables.